# Predicting Hospital Readmission in Diabetic Patients
## Notebook 5 of 7 - Support Vector Machine (Linear & RBF)

In this notebook I trained and evaluated **Support Vector Machine (SVM)** classifiers using two kernels. Rather than trying to classify every training point correctly, SVM focused specifically on finding the widest possible gap between classes - defined by the data points that sit closest to the boundary on either side (the support vectors).

- **Linear (LinearSVC):** Found the best straight-line boundary through the feature space. I used this for the full training set since it was designed for large datasets and ran significantly faster.
- **RBF (Radial Basis Function):** Mathematically transformed the data into a higher-dimensional space where a linear boundary could capture curved, non-linear patterns. Because RBF SVM was computationally expensive at scale, I trained all RBF models on a **10,000-row subsample** while evaluating on the full test set.

**Steps I followed:**
1. Loaded the data and re-derived the LASSO features (C=0.01, consistent with `3_FeatureSelection_LogisticRegression_PM.ipynb` and `4_RandomForest_PM.ipynb`)
2. Created the 10k subsample for RBF training
3. Ran a baseline LinearSVC with no class weighting
4. Swept C values across both feature sets for balanced LinearSVC and RBF SVM

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')  # suppresses convergence warnings common in SVM

## 2. Load Data

In [2]:
X_train_scaled = np.load('X_train_scaled.npy')
X_test_scaled  = np.load('X_test_scaled.npy')
y_train        = np.load('y_train.npy')
y_test         = np.load('y_test.npy')

feature_names = (
    pd.read_excel('diabetic_data_cleaned.xlsx')
      .drop(columns=['encounter_id', 'patient_nbr', 'readmitted'])
      .columns
)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set:     {X_test_scaled.shape}")

Training set: (71236, 98)
Test set:     (30530, 98)


## 3. Re-derive LASSO Feature Subset

I re-fit the same LASSO selector I used in `3_FeatureSelection_LogisticRegression_PM.ipynb` and `4_RandomForest_PM.ipynb` (C=0.01) to ensure all three models were compared on identical feature sets.

In [3]:
lasso_selector = LogisticRegression(
    l1_ratio=1, solver='saga', C=0.01,
    max_iter=1000, random_state=42,
    class_weight='balanced'
)
lasso_selector.fit(X_train_scaled, y_train)

surviving_features = np.where(np.any(lasso_selector.coef_ != 0, axis=0))[0]
print(f"LASSO features carried forward: {len(surviving_features)} / {X_train_scaled.shape[1]}")

X_train_lasso = X_train_scaled[:, surviving_features]
X_test_lasso  = X_test_scaled[:,  surviving_features]

LASSO features carried forward: 98 / 98


## 4. Create 10k Subsample for RBF Training

RBF SVM gets exponentially slower as the dataset grows, making it infeasible on the full 71k-row training set. I randomly sampled 10,000 rows and fixed the seed so the same rows were selected every time. The RBF models were always evaluated on the full test set - this is a methodological caveat I acknowledged when interpreting results in my final report.

In [4]:
np.random.seed(42)
sub_idx = np.random.choice(len(X_train_scaled), size=10000, replace=False)

X_train_sub       = X_train_scaled[sub_idx]
X_train_sub_lasso = X_train_lasso[sub_idx]
y_train_sub       = y_train[sub_idx]

print(f"RBF subsample: {len(y_train_sub):,} rows from {len(y_train):,} total")

RBF subsample: 10,000 rows from 71,236 total


## 5. Baseline LinearSVC (No Class Weighting)

As with every other unbalanced baseline in this project, the model learned that predicting the majority class most of the time was the easiest path to high accuracy, completely at the expense of Class 2 recall.

In [5]:
def run_baseline_svm():
    svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
    svm.fit(X_train_scaled, y_train)
    y_pred = svm.predict(X_test_scaled)

    print("BASELINE SVM (linear kernel, no class_weight, full features)")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred,
          target_names=['No Readmit (0)', '>30 Days (1)', '<30 Days (2)']))

run_baseline_svm()

BASELINE SVM (linear kernel, no class_weight, full features)
Accuracy: 0.5709
                precision    recall  f1-score   support

No Readmit (0)       0.58      0.91      0.71     16461
  >30 Days (1)       0.50      0.23      0.32     10644
  <30 Days (2)       0.00      0.00      0.00      3425

      accuracy                           0.57     30530
     macro avg       0.36      0.38      0.34     30530
  weighted avg       0.49      0.57      0.49     30530



## 6. Balanced LinearSVC - Sweep C × Feature Sets

I swept three C values across both the full and LASSO feature sets to show how the margin tradeoff affected results. A large C forced the boundary to hug the training data closely (risk of overfitting); a smaller C allowed some mistakes in exchange for a wider, more generalizable boundary.

In [6]:
def run_linear_svm(C, feature_set='full'):
    if feature_set == 'full':
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train_lasso, X_test_lasso
    label = f"LINEAR SVM | {feature_set} features | C={C}"

    svm = LinearSVC(C=C, class_weight='balanced', max_iter=2000, random_state=42)
    svm.fit(X_tr, y_train)
    y_pred = svm.predict(X_te)

    print(label)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred,
          target_names=['No Readmit (0)', '>30 Days (1)', '<30 Days (2)']))
    return svm, accuracy_score(y_test, y_pred)

In [7]:
C_values = [0.1, 1.0, 10.0]

print("LINEAR SVM (full training set)")
best_svm_linear, best_acc_linear = None, 0
for C in C_values:
    for fs in ['full', 'lasso']:
        svm_model, acc = run_linear_svm(C, feature_set=fs)
        if acc > best_acc_linear:
            best_acc_linear, best_svm_linear = acc, svm_model

print(f"\nBest Linear SVM accuracy: {best_acc_linear:.4f}")

LINEAR SVM (full training set)
LINEAR SVM | full features | C=0.1
Accuracy: 0.5604
                precision    recall  f1-score   support

No Readmit (0)       0.61      0.79      0.69     16461
  >30 Days (1)       0.48      0.35      0.40     10644
  <30 Days (2)       0.26      0.12      0.16      3425

      accuracy                           0.56     30530
     macro avg       0.45      0.42      0.42     30530
  weighted avg       0.53      0.56      0.53     30530

LINEAR SVM | lasso features | C=0.1
Accuracy: 0.5604
                precision    recall  f1-score   support

No Readmit (0)       0.61      0.79      0.69     16461
  >30 Days (1)       0.48      0.35      0.40     10644
  <30 Days (2)       0.26      0.12      0.16      3425

      accuracy                           0.56     30530
     macro avg       0.45      0.42      0.42     30530
  weighted avg       0.53      0.56      0.53     30530

LINEAR SVM | full features | C=1.0
Accuracy: 0.5604
                precis

## 7. Balanced RBF SVM - Sweep C × Feature Sets

The RBF kernel transformed the data into a higher-dimensional space where non-linear patterns became linearly separable. I set `cache_size=2000` to allocate 2 GB of RAM for caching kernel computations and speed up training. Note that this cell may take several minutes to run.

In [8]:
def run_rbf_svm(C, feature_set='full'):
    if feature_set == 'full':
        X_tr, X_te = X_train_sub, X_test_scaled
    else:
        X_tr, X_te = X_train_sub_lasso, X_test_lasso
    label = f"RBF SVM | {feature_set} features | C={C} | trained on 10k subsample"

    svm = SVC(
        kernel='rbf', C=C,
        class_weight='balanced',
        cache_size=2000,
        random_state=42
    )
    svm.fit(X_tr, y_train_sub)  # trained on the subsampled labels, not the full y_train
    y_pred = svm.predict(X_te)

    print(label)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred,
          target_names=['No Readmit (0)', '>30 Days (1)', '<30 Days (2)']))
    return svm, accuracy_score(y_test, y_pred)

In [9]:
print("RBF SVM (10k subsample)") #this cell may take several minutes
best_svm_rbf, best_acc_rbf = None, 0
for C in C_values:
    for fs in ['full', 'lasso']:
        svm_model, acc = run_rbf_svm(C, feature_set=fs)
        if acc > best_acc_rbf:
            best_acc_rbf, best_svm_rbf = acc, svm_model

RBF SVM (10k subsample) — this cell may take several minutes
RBF SVM | full features | C=0.1 | trained on 10k subsample
Accuracy: 0.4898
                precision    recall  f1-score   support

No Readmit (0)       0.63      0.61      0.62     16461
  >30 Days (1)       0.44      0.36      0.40     10644
  <30 Days (2)       0.18      0.30      0.23      3425

      accuracy                           0.49     30530
     macro avg       0.42      0.43      0.41     30530
  weighted avg       0.51      0.49      0.50     30530

RBF SVM | lasso features | C=0.1 | trained on 10k subsample
Accuracy: 0.4898
                precision    recall  f1-score   support

No Readmit (0)       0.63      0.61      0.62     16461
  >30 Days (1)       0.44      0.36      0.40     10644
  <30 Days (2)       0.18      0.30      0.23      3425

      accuracy                           0.49     30530
     macro avg       0.42      0.43      0.41     30530
  weighted avg       0.51      0.49      0.50     305

## 8. Best Results Summary

In [10]:
print("BEST RESULTS SUMMARY")
print(f"  Best Linear SVM accuracy: {best_acc_linear:.4f}")
print(f"  Best RBF SVM accuracy:    {best_acc_rbf:.4f}")

BEST RESULTS SUMMARY
  Best Linear SVM accuracy: 0.5604
  Best RBF SVM accuracy:    0.4910


## 9. Interpretation

The most notable result across all SVM configurations was the **Linear vs. RBF tradeoff**:

| Model | Accuracy | Class 2 Recall |
|---|---|---|
| LinearSVC Baseline | 0.5709 | 0.00 |
| LinearSVC Balanced C=0.1 | 0.5604 | 0.12 |
| RBF SVM C=0.1 (10k sub) | 0.4898 | **0.30** |

The RBF kernel achieved the **highest Class 2 recall of any model in the entire project** (0.30) while producing the lowest overall accuracy (0.49). The non-linear boundary that RBF drew was apparently better at capturing the information that distinguished early readmission patients, even at the cost of more overall misclassifications. In a clinical setting where the primary goal was to flag patients most at risk of coming back within 30 days, that higher recall was the more meaningful result - one in three early readmission patients flagged before discharge was meaningfully better than zero, which every unbalanced model had produced.